# Trabalho Prático 1 - INF01017
## Análise Exploratória de Dados (EDA)
### Dataset: Ames Mutagenicity

**Objetivo:** Realizar análise exploratória completa do dataset para compreender suas características, distribuições e padrões.

---

## 1. Imports e Configurações

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Seed para reprodutibilidade
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 2. Carregamento dos Dados

In [ ]:
# Carregar dataset
df = pd.read_csv('../data/raw/ames_mutagenicity_data.csv')

print(f"Shape do dataset: {df.shape}")
print(f"Número de amostras: {df.shape[0]:,}")
print(f"Número de features: {df.shape[1]}")

In [ ]:
# Primeiras linhas
df.head()

## 3. Informações Gerais do Dataset

In [ ]:
# Informações sobre tipos de dados e memória
df.info()

In [ ]:
# Identificar diferentes tipos de colunas
metadata_cols = ['Id', 'Name', 'CAS', 'SMILES RDKit', 'Partition']
strain_cols = ['TA98', 'TA100', 'TA102', 'TA1535', 'TA1537']
target_col = 'Overall'

feature_cols = [col for col in df.columns 
               if col not in metadata_cols + strain_cols + [target_col]]

print(f"Colunas de metadados: {len([c for c in metadata_cols if c in df.columns])}")
print(f"Colunas de cepas: {len([c for c in strain_cols if c in df.columns])}")
print(f"Coluna alvo: {target_col in df.columns}")
print(f"Colunas de features (descritores moleculares): {len(feature_cols)}")

## 4. Análise da Variável Alvo (Target)

In [ ]:
# Distribuição da variável alvo
target_distribution = pd.DataFrame({
    'Count': df[target_col].value_counts(),
    'Percentage': df[target_col].value_counts(normalize=True) * 100
})

print("Distribuição da variável alvo (Overall):")
print(target_distribution)
print(f"\nClasse 0 (não-mutagênico): {target_distribution.loc[0, 'Percentage']:.2f}%")
print(f"Classe 1 (mutagênico): {target_distribution.loc[1, 'Percentage']:.2f}%")

In [ ]:
# Visualização da distribuição
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico de barras
target_distribution['Count'].plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Distribuição das Classes - Contagem', fontsize=14)
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Contagem')
axes[0].grid(True, alpha=0.3)

# Gráfico de pizza
axes[1].pie(target_distribution['Count'], labels=target_distribution.index, 
           autopct='%1.1f%%', startangle=90, colors=['steelblue', 'coral'])
axes[1].set_title('Distribuição das Classes - Proporção', fontsize=14)

plt.tight_layout()
plt.show()

## 5. Análise de Valores Faltantes

In [ ]:
# Valores faltantes
missing_values = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100
})

missing_values = missing_values[missing_values['Missing_Count'] > 0].sort_values(
    'Missing_Percentage', ascending=False
)

if len(missing_values) > 0:
    print(f"Total de colunas com valores faltantes: {len(missing_values)}")
    print("\nTop 10 colunas com mais valores faltantes:")
    print(missing_values.head(10))
else:
    print("✓ Nenhum valor faltante encontrado no dataset!")

## 6. Estatísticas Descritivas das Features

In [ ]:
# Estatísticas descritivas
df[feature_cols].describe()

## 7. Análise de Correlação

In [ ]:
# Correlação das features com o target
correlations = df[feature_cols].corrwith(df[target_col]).abs().sort_values(ascending=False)

print("Top 20 features mais correlacionadas com o target:")
print(correlations.head(20))

In [ ]:
# Visualização das top correlações
fig, ax = plt.subplots(figsize=(10, 8))
top_corr = correlations.head(20)
top_corr.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 20 Features Mais Correlacionadas com o Target', fontsize=14)
ax.set_xlabel('Correlação Absoluta')
ax.set_ylabel('Feature')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Distribuição de Features

In [ ]:
# Visualizar distribuição de algumas features aleatórias
selected_features = np.random.choice(feature_cols, size=9, replace=False)

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.ravel()

for idx, feature in enumerate(selected_features):
    axes[idx].hist(df[feature].dropna(), bins=50, color='steelblue', alpha=0.7)
    axes[idx].set_title(f'{feature}', fontsize=10)
    axes[idx].set_xlabel('Valor')
    axes[idx].set_ylabel('Frequência')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Análise de Duplicatas

In [ ]:
# Verificar duplicatas
n_duplicates = df.duplicated().sum()
print(f"Número de linhas duplicadas: {n_duplicates}")

if n_duplicates > 0:
    print("\nPrimeiras linhas duplicadas:")
    print(df[df.duplicated()].head())

## 10. Conclusões da EDA

**Escreva aqui as principais conclusões da análise exploratória:**

1. Distribuição das classes (balanceamento)
2. Qualidade dos dados (missing values, duplicatas)
3. Características das features (escala, variância)
4. Features mais relevantes
5. Necessidades de pré-processamento identificadas

---